In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
cd drive/MyDrive/Multi-turn-RAG/

/content/drive/MyDrive/Multi-turn-RAG


In [4]:
pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 979.4/979.4 kB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.7/508.7 kB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.7/627.7 kB 60.3 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28


In [5]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Any

# =========================
# 프로젝트 루트 경로
# =========================
# 현재 작업 중인 폴더를 프로젝트 루트로 사용
BASE_DIR = Path.cwd()
sys.path.insert(0, str(BASE_DIR))

# =========================
# 프로젝트 내부 모듈 import
# =========================
from src.pipeline import build_graph
from src.memory import reset_all
from src.retriever import retrieve, hybrid_rrf_search

TOP_K = 5

In [6]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"[WARN] JSON decode error in {path} line {line_num}: {e}")
    return rows


def dump_jsonl(rows: list[dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def parse_query_id(query_id: str) -> tuple[str, int]:
    if "<::>" not in query_id:
        raise ValueError(f"Unexpected query-id format: {query_id}")
    session_key, turn_str = query_id.split("<::>", 1)
    return session_key, int(turn_str)


def sort_key(row: dict[str, Any]) -> tuple[str, int]:
    session_key, turn_no = parse_query_id(row["query-id"])
    return session_key, turn_no


def build_session_id(session_key: str, turn_type: str, turn_no: int) -> str:
    if turn_type == "multi-turn":
        return session_key
    elif turn_type == "single-turn":
        return f"{session_key}__turn_{turn_no}"
    raise ValueError(f"Unknown turn_type: {turn_type}")


def require_hit_field(hit: dict[str, Any], field_name: str, search_type: str) -> Any:
    if field_name not in hit:
        raise KeyError(
            f"[{search_type}] 검색 결과 hit에 '{field_name}' 필드가 없습니다. "
            f"src/retriever.py에서 반환하도록 맞춰야 합니다. hit keys={list(hit.keys())}"
        )
    return hit[field_name]


def get_retrieved_docs(query: str, search_type: str, top_k: int = TOP_K) -> list[dict[str, Any]]:
    if search_type == "hybrid":
        hits = hybrid_rrf_search(
            query=query,
            top_k=max(10, top_k * 2),
            final_k=top_k,
        )

        docs: list[dict[str, Any]] = []
        for rank, hit in enumerate(hits, start=1):
            docs.append(
                {
                    "chunk_id": require_hit_field(hit, "doc_id", search_type),
                    "rank": rank,
                    "score": {
                        "hybrid_score": hit.get("rrf_score"),
                        "dense_score_raw": hit.get("dense_score_raw"),
                        "bm25_score_raw": hit.get("bm25_score_raw"),
                    },
                    "text": require_hit_field(hit, "text", search_type),
                }
            )
        return docs

    if search_type in {"bm25", "dense"}:
        hits = retrieve(query, mode=search_type, top_k=top_k)

        docs: list[dict[str, Any]] = []
        for rank, hit in enumerate(hits, start=1):
            docs.append(
                {
                    "chunk_id": require_hit_field(hit, "doc_id", search_type),
                    "rank": rank,
                    "score": hit.get("score"),
                    "text": require_hit_field(hit, "text", search_type),
                }
            )
        return docs

    raise ValueError(f"Unsupported search_type: {search_type}")


def run_one_turn(app, question: str, session_id: str, search_type: str, top_k: int = TOP_K):
    config = {"configurable": {"thread_id": session_id}}
    out = app.invoke({"input": question}, config=config)

    messages = out.get("messages", [])
    if messages:
        last_msg = messages[-1]
        generated_answer = getattr(last_msg, "content", str(last_msg))
    else:
        generated_answer = ""

    rewritten_query = out.get("rewritten_query")
    search_query = rewritten_query if rewritten_query else question
    retrieved_docs = get_retrieved_docs(search_query, search_type=search_type, top_k=top_k)

    return generated_answer, rewritten_query, retrieved_docs


def enrich_records(
    records: list[dict[str, Any]],
    turn_type: str,
    search_type: str,
    top_k: int = TOP_K,
) -> list[dict[str, Any]]:
    original_order = [row["query-id"] for row in records]
    result_map: dict[str, dict[str, Any]] = {}

    sorted_records = sorted(records, key=sort_key)
    app = build_graph(retrieval_mode=search_type, top_k=top_k)

    reset_all()
    current_session_key: str | None = None

    for idx, row in enumerate(sorted_records, start=1):
        query_id = row["query-id"]
        question = row["question"]
        session_key, turn_no = parse_query_id(query_id)

        if turn_type == "multi-turn":
            if current_session_key != session_key:
                reset_all()
                current_session_key = session_key
            session_id = build_session_id(session_key, turn_type, turn_no)

        elif turn_type == "single-turn":
            reset_all()
            current_session_key = None
            session_id = build_session_id(session_key, turn_type, turn_no)

        else:
            raise ValueError(f"Unknown turn_type: {turn_type}")

        generated_answer, rewritten_query, retrieved_docs = run_one_turn(
            app=app,
            question=question,
            session_id=session_id,
            search_type=search_type,
            top_k=top_k,
        )

        enriched = dict(row)  # 기존 row 전체 유지
        enriched["turn_type"] = turn_type
        enriched["search_type"] = search_type
        enriched["session_id"] = session_id
        enriched["generated_answer"] = generated_answer
        enriched["rewritten_query"] = rewritten_query if turn_type == "multi-turn" else None
        enriched["retrieved_docs"] = retrieved_docs

        result_map[query_id] = enriched

        print(
            f"[{idx}/{len(sorted_records)}] "
            f"query-id={query_id} | turn_type={turn_type} | search_type={search_type}"
        )

    reset_all()
    return [result_map[qid] for qid in original_order]

In [11]:
INPUT_PATH = BASE_DIR / "data" / "processed" / "question_with_targets.jsonl"

# 예시:
# "single-turn" or "multi-turn"
TURN_TYPE = "single-turn"

# "bm25" or "dense" or "hybrid"
SEARCH_TYPE = "dense"

# top-5 문서 저장
TOP_K = 5

# 테스트면 숫자 넣고, 전체 돌릴 거면 None
LIMIT = 10

OUTPUT_PATH = BASE_DIR / "outputs" / "results" / f"question_with_targets__{TURN_TYPE}__{SEARCH_TYPE}__limit{LIMIT}.jsonl"

print("INPUT_PATH :", INPUT_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH)
print("TURN_TYPE  :", TURN_TYPE)
print("SEARCH_TYPE:", SEARCH_TYPE)
print("TOP_K      :", TOP_K)
print("LIMIT      :", LIMIT)

INPUT_PATH : /content/drive/MyDrive/Multi-turn-RAG/data/processed/question_with_targets.jsonl
OUTPUT_PATH: /content/drive/MyDrive/Multi-turn-RAG/outputs/results/question_with_targets__single-turn__dense__limit10.jsonl
TURN_TYPE  : single-turn
SEARCH_TYPE: dense
TOP_K      : 5
LIMIT      : 10


In [12]:
records = load_jsonl(INPUT_PATH)
print("loaded rows:", len(records))

if LIMIT is not None:
    records = records[:LIMIT]
    print("test rows:", len(records))

enriched = enrich_records(
    records=records,
    turn_type=TURN_TYPE,
    search_type=SEARCH_TYPE,
    top_k=TOP_K,
)

dump_jsonl(enriched, OUTPUT_PATH)
print(f"[DONE] saved: {OUTPUT_PATH}")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


loaded rows: 180
test rows: 10
[Skip] First turn, no rewriting:  Alternative means of salary for my employees
[Search Query] (dense)  Alternative means of salary for my employees


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[1/10] query-id=1c74814752c3f9d4ed0b99c16e1ed192<::>1 | turn_type=single-turn | search_type=dense
[Skip] First turn, no rewriting:  Do employers need to report their employee's salaries and withhold money for income tax purposes?
[Search Query] (dense)  Do employers need to report their employee's salaries and withhold money for income tax purposes?


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[5/10] query-id=1c74814752c3f9d4ed0b99c16e1ed192<::>5 | turn_type=single-turn | search_type=dense
[Skip] First turn, no rewriting:  Is personal income tax the same as corporate tax?
[Search Query] (dense)  Is personal income tax the same as corporate tax?


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[6/10] query-id=1c74814752c3f9d4ed0b99c16e1ed192<::>6 | turn_type=single-turn | search_type=dense
[Skip] First turn, no rewriting:  Investments?
[Search Query] (dense)  Investments?


KeyboardInterrupt: 

In [10]:
preview_rows = load_jsonl(OUTPUT_PATH)

print("saved rows:", len(preview_rows))
print(json.dumps(preview_rows[6], ensure_ascii=False, indent=2))

saved rows: 10
{
  "query-id": "1c74814752c3f9d4ed0b99c16e1ed192<::>7",
  "question": " Investments?",
  "corpus-id": [
    "321114-0-59",
    "315105-0-1161",
    "443354-0-393"
  ],
  "text": [
    "\nIf you receive dividends on an investment, those are taxed.",
    "\nNot exactly. There are a few ways to manage your taxes with investments. 1) For most investments you get taxed on any gain in value in the investment or dividends paid by that investment. Most investments (with some exceptions for mutual funds) you don't take the tax hit until you sell the investment and realize the gain. For bonds, cds, and other cash type investments you have to pay taxes in the year they pay out the interest or dividend. 2) You can put money (up to a certain limit) in a traditional IRA and can subtract that amount from your income for tax calculation for the year you invest it. However, you are going to pay taxes on it when you take the money out at retirement. It really just delays the taxes. 3) If